In [2]:
!pip install playwright

   ---------------------------------------- 0.0/37.9 MB ? eta -:--:--
   ------ --------------------------------- 5.8/37.9 MB 29.7 MB/s eta 0:00:02
   --------------------- ------------------ 20.2/37.9 MB 49.2 MB/s eta 0:00:01
   ---------------------------------------  37.7/37.9 MB 64.4 MB/s eta 0:00:01
   ---------------------------------------- 37.9/37.9 MB 54.1 MB/s eta 0:00:00

   -------------------- ------------------- 1/2 [playwright]
   -------------------- ------------------- 1/2 [playwright]
   -------------------- ------------------- 1/2 [playwright]
   -------------------- ------------------- 1/2 [playwright]
   -------------------- ------------------- 1/2 [playwright]
   ---------------------------------------- 2/2 [playwright]



Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/149.0.0.0 Safari/537.36

In [8]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import re

URL = "https://m.market09.kr/best"
MAX_PRODUCTS = 10

def to_int(text):
    if not text:
        return 0
    nums = re.sub(r"[^\d]", "", str(text))
    return int(nums) if nums else 0

headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/149.0.0.0 Safari/537.36"
}

res = requests.get(URL, headers=headers, timeout=20)
print("상태코드:", res.status_code)

soup = BeautifulSoup(res.text, "html.parser")

products = soup.select("figure.product-list-item")[:MAX_PRODUCTS]

print("상품 수:", len(products))

rows = []

for product in products:
    img = product.select_one("img")
    detail = product.select_one("figcaption.detail-info")

    product_name = img.get("alt", "") if img else ""
    image_url = img.get("src", "") if img else ""
    description = detail.get_text(" ", strip=True) if detail else ""

    rows.append({
        "id": "",
        "category": "",
        "product_name": product_name,
        "image_url": image_url,
        "original_price": 0,
        "group_price": 0,
        "discount_rate": 0,
        "recruit_count": 0,
        "current_count": 0,
        "deadline": "",
        "description": description
    })

df = pd.DataFrame(rows)
df.to_csv("market09_test.csv", index=False, encoding="utf-8-sig")

df

상태코드: 200
상품 수: 0


""


In [10]:
import requests
import pandas as pd

url = "https://new.market09.kr/v2/get_bests.php"

res = requests.post(url, timeout=20)
data = res.json()

products = data.get("contents", [])[:10]

rows = []

for p in products:
    price = p.get("dscn_idct", {}) or {}
    group = p.get("grup_buy_info", {}) or {}

    rows.append({
        "id": p.get("idx", ""),
        "category": p.get("gods_dvsn_cd", ""),
        "product_name": p.get("subject", ""),
        "image_url": p.get("image", ""),
        "original_price": int(price.get("base_price") or 0),
        "group_price": int(group.get("gb_price") or price.get("sale_price") or 0),
        "discount_rate": int(price.get("rate") or group.get("gb_price_rate") or 0),
        "recruit_count": int(group.get("limt_cnt") or 0),
        "current_count": int(p.get("total_sale_count") or 0),
        "deadline": group.get("gb_end_dtm", ""),
        "description": p.get("context", "") or p.get("context2", "")
    })

df = pd.DataFrame(rows)

df.to_csv("market09_best_test.csv", index=False, encoding="utf-8-sig")
df

,id,category,product_name,image_url,original_price,group_price,discount_rate,recruit_count,current_count,deadline,description
0,13035769,01,[산앤들산삼] 강원도 자연이 키운 4-5년근 야생 장뇌삼 50뿌리/산양산삼,https://gcdn.market09.kr/data/image/4068041_01...,120000,29900,75,2,2216,2028-04-11 02:30:01,
1,13029001,01,★참외 10kg ★ 성주 꿀 참외 산지직송 고당도 10kg 노마진 초특가 가정용 혼합과,https://gcdn.market09.kr/data/image/4051992_01...,50000,17900,64,2,508,2028-03-11 02:30:02,
2,13112988,01,프로코 고농축 8종효소 라벤더 세탁세제 2.5L 2+2 (4개입),https://gcdn.market09.kr/data/image/4202823_01...,28900,10900,62,2,2127,2027-04-04 16:20:59,
3,13040476,01,★참외 3kg 특가★ 성주 꿀 참외 산지직송 고당도 2026년 첫 출하 가정용 과수랜덤,https://gcdn.market09.kr/data/image/4073967_01...,100000,9900,90,2,14916,2028-05-02 02:30:01,
4,13204570,01,[연세우유] 편의점 인기 연세 콜드브루 커피우유 멸균우유 24팩/48팩/96팩,https://gcdn.market09.kr/data/image/4296591_01...,19900,10900,45,2,494,2028-05-14 15:25:59,
5,10008718,01,기분좋은선택 프리미엄 화장지 (27M*30롤) (100% 천연펄프) (추가할인),https://gcdn.market09.kr/data/image/7113d55bc4...,31800,11400,64,2,241909,2028-06-07 02:30:01,온 가족을 생각한 프리미엄 화장지!\r\n매일매일 부드럽고 편안하게~\r\n\r\n
6,13204631,01,나폴리가든 남성 캐쥬얼 벨트,https://gcdn.market09.kr/data/image/4294958_01...,20000,7500,63,2,187,2028-05-14 18:12:59,
7,13134204,01,레이시스 여성 남성 젤리 슬리퍼 샌들 아쿠아슈즈 EVA 여름 쿠션 물놀이 빅사이즈 ...,https://gcdn.market09.kr/data/image/4227165_01...,41800,19700,53,2,268,2027-07-11 18:13:59,
8,13055310,01,[깨끗한나라] 3겹 데코엠보 더 프라임 롤화장지 25m 30롤 x 2팩,https://gcdn.market09.kr/data/image/4124916_01...,50000,27300,45,2,190,2026-07-10 10:12:59,
9,13199874,01,★TV출연농가★19브릭스 성주 참외 3kg 역마진초특가(+과일세척제 증정),https://gcdn.market09.kr/data/image/4285764_01...,30000,9900,67,2,2263,2028-04-22 18:26:59,


In [11]:
import json

print(json.dumps(products[0], ensure_ascii=False, indent=2))

{
  "type": "product",
  "rank": "01",
  "price_type": "0",
  "state": "0",
  "idx": "13035769",
  "subject": "[산앤들산삼] 강원도 자연이 키운 4-5년근 야생 장뇌삼 50뿌리/산양산삼",
  "soldout": "0",
  "glbl_yn": "N",
  "adlt_yn": "N",
  "gods_dvsn_cd": "01",
  "status": "0",
  "image": "https://gcdn.market09.kr/data/image/4068041_01_5ef0296714bd4f3798d8053b52e480f6.jpg",
  "image_sub": "https://gcdn.market09.kr/data/image/4068041_02_7261f9cbfa0865f72bb1ac5c3336856a.jpg",
  "context": "",
  "context2": "",
  "btn_text": "구매하기",
  "copn_use_yn": "Y",
  "rsmn_use_yn": "Y",
  "gods_img": {
    "list": [],
    "count": 0
  },
  "brnd_info": {},
  "xprice_date_tooltip": "2024.04.23 기준\r\n* 11번가, G마켓, 옥션, 인터파크, GS SHOP,\r\n롯데ON, SSG, CJ온스타일, 티몬 \r\n[위 9개 쇼핑몰 중 최저가격 등재가 (쿠폰 적용가 제외)]",
  "total_sale_count": 2216,
  "total_sale_count_label": "2,216개 구매",
  "total_sale_count_label_rev": "구매 2,216",
  "rdct_befr_tooltip": "0000.00.00 기준\r\n인하 전 판매가와의 차액을 % 로 제공하는 정보",
  "time_sale_gods_yn": "N",
  "timesale_start_dtm": "",